# CineScore Phase 1C: Macro-Economic Normalization
### Anchoring Historical Data to a 2026 USD Baseline (1.00)

**Objective:** Normalize all financial metrics to a single temporal baseline (2026 purchasing power).

In [ ]:
import pandas as pd
import numpy as np
import os

if os.path.exists('/content'):
    INPUT_PATH = "/content/drive/MyDrive/CineScore/v8_FINAL_CINESCORE_NLP.csv"
    OUTPUT_PATH = "/content/drive/MyDrive/CineScore/v2_economically_normalized_df.csv"
else:
    INPUT_PATH = "../Data/Processed_Dataset/v8_FINAL_CINESCORE_NLP.csv"
    OUTPUT_PATH = "../Data/Processed_Dataset/v2_economically_normalized_df.csv"

print(f"Loading API Enriched Dataset: {INPUT_PATH}")
df = pd.read_csv(INPUT_PATH)
print(f"Dataset Shape: {df.shape}")

In [ ]:
cpi_map = {
    1970: 0.11, 1980: 0.24, 1990: 0.38, 2000: 0.50,
    2010: 0.63, 2015: 0.69, 2020: 0.76, 2021: 0.79,
    2022: 0.86, 2023: 0.89, 2024: 0.93, 2025: 0.97,
    2026: 1.00
}

def get_cpi_multiplier(year):
    if pd.isna(year): return 1.0
    anchors = sorted(cpi_map.keys())
    values = [cpi_map[a] for a in anchors]
    return np.interp(year, anchors, values)

if 'release_date' in df.columns and 'release_year' not in df.columns:
    df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
    df['release_year'] = df['release_date'].dt.year

In [ ]:
df['cpi_multiplier'] = df['release_year'].apply(get_cpi_multiplier)
df['inflated_budget'] = df['budget'] / df['cpi_multiplier']
df['inflated_revenue'] = df['revenue'] / df['cpi_multiplier']
df['log_inflated_budget'] = np.log1p(df['inflated_budget'])
df['log_inflated_revenue'] = np.log1p(df['inflated_revenue'])

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)
print("🔥 Macro-Economic Normalization Complete 🔥")
print(f"Artifact finalized to: {OUTPUT_PATH}")